In [19]:
import os
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    RunResult,
    OpenAIChatCompletionsModel,
    ModelSettings,
    function_tool,
)

import asyncio
from datetime import datetime
from typing import Literal

from agents.tracing import set_tracing_disabled

In [20]:
local_model = OpenAIChatCompletionsModel(
    model="qwen2.5-coder:3b",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [21]:
from agents import Agent, Runner

# BAD instructions — too vague
bad_agent = Agent(
    name="Bad Agent",
    instructions="You are helpful.",
    model=local_model
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    model=local_model,
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Score: 7/10
Issues: []
Suggestions: []


In [22]:
from agents import Agent, Runner, ModelSettings

# Creative agent — high temperature
creative_agent = Agent(
    model=local_model,
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    model=local_model,
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


 CREATIVE:
 In the heart of vibrant Lahore, amidst a kaleidoscope of colors and bustling markets, stands the Badshahi Mosque - a masterpiece of Islamic architecture that has captured the imagination and hearts of people for generations. This grand edifice is not just a building but an architectural marvel, a te

 PRECISE:
 The Badshahi Mosque is a significant Islamic architectural masterpiece located in Lahore, Pakistan. It was built by Emperor Aurangzeb between 1673 and 1680 as the new capital of the Mughal Empire. Here are some key details about the mosque:

1. Location: The mosque stands on the site of an earlier s


In [23]:
from pydantic import BaseModel, Field


# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    model=local_model,
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

OPENAI_API_KEY is not set, skipping trace export


Title:          Lex Fridman Podcast
Host:           Lex Fridman
Rating:         4.0/10
Genre:          Technology
Tech Level:     Intermediate
Summary:        'Lex Fridman Podcast' is a comprehensive AI, machine learning, and technology show hosted by Lex Fridman. The podcast covers everything from theoretical concepts to practical applications across various fields.
Recommended:    Yes


OPENAI_API_KEY is not set, skipping trace export


In [24]:
class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        description="Confidence score 0.0-1.0"
    )


classifier = Agent(
    model=local_model,
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1)
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")

OPENAI_API_KEY is not set, skipping trace export



Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: budget
   Search: 'laptop for school budget' | Confidence: 9000%


OPENAI_API_KEY is not set, skipping trace export



Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: low | Price: mid
   Search: 'birthday gifts books novels' | Confidence: 9000%


OPENAI_API_KEY is not set, skipping trace export



Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: mid
   Search: 'phone charger broken' | Confidence: 9000%


OPENAI_API_KEY is not set, skipping trace export


In [25]:

def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    model=local_model,
    name="Smart Scheduler",
    instructions=dynamic_instructions  # <-- Pass function, not string!
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Based on the current date/time of 2026-04-09 16:11, it's afternoon. Since it's not too late and you haven't had a meeting or collaboration planned for today, you might consider arranging one in the next few hours or discussing your goals with team members during a brief break if that makes sense within your work schedule.

Enjoy a productive afternoon!


OPENAI_API_KEY is not set, skipping trace export


In [26]:

class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    model=local_model,
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2)
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")

OPENAI_API_KEY is not set, skipping trace export



Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: negative
Priority:  P0-critical
Route to:  technical
Summary:   API error causing service outage
Response:  I'm sorry to hear that our API is currently experiencing issues, which has resulted in a service outage. We understand the importance of your applicat...


OPENAI_API_KEY is not set, skipping trace export



Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: neutral
Priority:  P1-high
Route to:  billing
Summary:   Billing issue: double charge detected.
Response:  Hello! I'm sorry to hear that you're experiencing a billing issue with your account. Could you please provide me with some more details about the char...


OPENAI_API_KEY is not set, skipping trace export



Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P3-low
Route to:  general
Summary:   Customer feedback for adding dark mode to the product.
Response:  Thank you for your kind words! We appreciate your feedback and are always looking to improve our products. Dark mode is a feature we've been consideri...


OPENAI_API_KEY is not set, skipping trace export
